# Class 02 · Part 1: Getting station data

Before we can plot anything, we need the data. In this notebook you learn

1. **where** the measurements of the German weather stations are, and what the files are called,
2. how to **find** a station and its file,
3. how to **download** it with Python, and
4. why it is good to put these steps into a **function**.

In part 2, `class02_use_station_data.ipynb`, you read the data and plot it.

Run the cells from top to bottom with `Shift + Enter`.

## 1. Where is the data?

The German Weather Service (DWD) publishes the measurements of its stations for free on its open data server.
Open this link in a new tab of your browser and look around: it is a list of folders and files, like the file
browser of JupyterLab.

**https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/**

The folder `kl` holds the daily **climate** values (*Klima*) of all stations. It has two folders:

- `historical/`: every station from its first day until the end of last year. **We use this one.**
- `recent/`: only the last months, updated every day.

Open `historical/`. There is one zip file per station, and its name tells you what is inside:

```
tageswerte_KL_01975_19360101_20251231_hist.zip
│          │  │     │                 └ historical
│          │  │     └ period: from 1 January 1936 to 31 December 2025
│          │  └ station number: 01975 is Hamburg-Fuhlsbüttel
│          └ KL: climate values
└ tageswerte: daily values
```

You could click on a file to download it, unzip it on your laptop and upload it to JupyterLab. For one file
that works. With Python it is a few lines, and they work for every station.

## 2. Find a station

The folder also has the **list of stations**, `KL_Tageswerte_Beschreibung_Stationen.txt`: number, period,
height, position and name of every station.

`requests` asks a web server for a page or a file, as your browser does. `.content` is what the server sent
back. The list is written with German letters (ä, ö, ü), so we turn it into text with `.decode("latin1")`.

In [ ]:
import requests

historical = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/"

response = requests.get(historical + "KL_Tageswerte_Beschreibung_Stationen.txt")
stations = response.content.decode("latin1")

# the first line: the names of the columns
print(stations.splitlines()[0])

The list has more than 1000 lines. A loop prints only those with "Hamburg" (in the name or in the state):

In [ ]:
for line in stations.splitlines():
    if "Hamburg" in line:
        print(line.rstrip())  # rstrip: without the spaces at the end of the line

The first column is the station number. **Hamburg-Fuhlsbüttel is 01975**, with data since 1936.

## 3. Find the file of the station

The period is part of the file name, and it changes every year. So instead of typing the name, we look for it
in the list of files of the folder. The server sends that list as a web page (HTML); in it, every file name
is written between quotes:

```
<a href="tageswerte_KL_01975_19360101_20251231_hist.zip">tageswerte_KL_01975_...
```

In [ ]:
listing = requests.get(historical).text

# cut the page at every quote ", and keep the piece that is the file of station 01975
for part in listing.split('"'):
    if part.startswith("tageswerte_KL_01975_") and part.endswith("_hist.zip"):
        file_name = part

file_name

## 4. Download it

Now we ask the server for the file itself and save it in the folder `data`, next to this notebook.
`pathlib.Path` is a path to a file or folder: `.mkdir` creates a folder, `.write_bytes` saves a file.

In [ ]:
import pathlib

response = requests.get(historical + file_name)

folder = pathlib.Path("data")
folder.mkdir(exist_ok=True)  # exist_ok=True: no error if the folder is already there
(folder / file_name).write_bytes(response.content)

print("saved", folder / file_name, len(response.content) / 1e6, "MB")

Look in the file browser on the left: `data` now has the zip file of Hamburg.

## 5. Why a function?

For Berlin we need the same steps again, only with another station number. Copying the cells and changing
the number in three places works, but it is easy to forget one place, and the notebook gets long.

A **function** gives the steps a name. You learned it in class 01:

- `def name(inputs):` starts it; the inputs are called **parameters**,
- the indented lines are the steps,
- `return` gives back the result.

Here is a small function for step 3: it finds the file name of any station.

In [ ]:
def find_file_name(station):
    for part in listing.split('"'):
        if part.startswith(f"tageswerte_KL_{station}_") and part.endswith("_hist.zip"):
            return part


print(find_file_name("01975"))  # Hamburg-Fuhlsbüttel
print(find_file_name("00433"))  # Berlin-Tempelhof

One line per station, and the steps are written only once: if something changes, you change it in one
place. That is why functions are good: **less repetition, fewer mistakes, easier to read**.

## 6. The function of the course: `download_dwd_daily`

The course has a function that does steps 3 and 4 together, in `utils/` next to this notebook. You load it
with `import`, like a package. `help` shows its description (the text between `"""` at its start):

In [ ]:
from utils import download_dwd_daily

help(download_dwd_daily)

And this is its code. Read it: you know every step from sections 3 and 4. It does one more thing first:
if the file is already in the folder, it does not download it again (faster, and kinder to the server).
In JupyterLab you can also see the code of any function with `download_dwd_daily??`.

In [ ]:
import inspect

print(inspect.getsource(download_dwd_daily))

Two details:

- `f"{int(station_id):05d}"` writes the number with 5 digits, as in the file names: `1975` becomes `"01975"`.
  So you can write `download_dwd_daily(1975)` or `download_dwd_daily("01975")`.
- `raise ValueError(...)` stops with a clear message if the station has no file.

Now each station is one line. Hamburg is already in `data`, so nothing is downloaded; Berlin is new:

In [ ]:
hamburg_file = download_dwd_daily(1975)
berlin_file = download_dwd_daily(433)

print(hamburg_file)
print(berlin_file)

## 7. Exercise

1. Search the list of stations (section 2) for a place you know, e.g. near the region of your page of class 01
   if it is in Germany. Note its number.
2. Download its data with `download_dwd_daily` and check that the file is in `data`.
3. Try a number that does not exist, e.g. `download_dwd_daily(99999)`. What does the message say?

In [ ]:
# 1. search the list of stations

In [ ]:
# 2. download the data of your station

In [ ]:
# 3. a station that does not exist

## Next

Open **`class02_use_station_data.ipynb`**: you read the data of Hamburg with pandas and plot its temperature.